<a href="https://colab.research.google.com/github/phamvanhung33182005-ux/Parameter_Identification_of_PMSM_using_PINN/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===== CELL 1: KẾT NỐI DRIVE VÀ ĐỌC 5 DATASET =====

from google.colab import drive  # Import công cụ kết nối Google Drive với Google Colab.
import os  # Import thư viện os để xử lý đường dẫn.
import pandas as pd  # Import pandas để đọc và xử lý dữ liệu bảng.
import numpy as np  # Import numpy để xử lý dữ liệu số.

drive.mount('/content/drive')  # Gắn Google Drive vào môi trường Colab.

DRIVE_PROJECT = '/content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN'  # Khai báo thư mục gốc của project.

RAW_DATA_DIR = os.path.join(DRIVE_PROJECT, 'dataset', 'raw')  # Tạo đường dẫn đến thư mục chứa dataset raw.

PROCESSED_DATA_DIR = os.path.join(DRIVE_PROJECT, 'dataset', 'processed')  # Tạo đường dẫn đến thư mục lưu dataset đã preprocessing.

DATA_FILES = ['Data1.csv', 'Data2.csv', 'Data3.csv', 'Data4.csv', 'Data5.csv']  # Khai báo tên 5 dataset.

INPUT_COLUMNS = ['ud', 'uq', 'Id', 'Iq', 'nm']  # Khai báo 5 biến input của mô hình.

TARGET_COLUMNS = ['Rs', 'Ld', 'Lq']  # Khai báo 3 tham số cần nhận dạng.

TIME_COLUMN = 'Time'  # Khai báo cột thời gian được giữ lại cho PINN.

TRAIN_FILES = ['Data1.csv', 'Data2.csv', 'Data3.csv']  # Khai báo 3 dataset dùng để train.

VALIDATION_FILES = ['Data4.csv']  # Khai báo dataset dùng để validation.

TEST_FILES = ['Data5.csv']  # Khai báo dataset dùng để test.

os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)  # Tạo thư mục processed nếu thư mục chưa tồn tại.

datasets = {}  # Tạo dictionary để lưu 5 DataFrame.

for file_name in DATA_FILES:  # Lặp qua từng dataset.
    file_path = os.path.join(RAW_DATA_DIR, file_name)  # Tạo đường dẫn đầy đủ đến file CSV.
    datasets[file_name] = pd.read_csv(file_path)  # Đọc file CSV vào DataFrame.

print('Đã đọc:', len(datasets), 'dataset')  # In số lượng dataset đã đọc.

print('Thư mục processed:', PROCESSED_DATA_DIR)  # In đường dẫn thư mục sẽ lưu dữ liệu đã xử lý.

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã đọc: 5 dataset
Thư mục processed: /content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN/dataset/processed


In [ ]:
# ===== CELL 2: TÁCH TRAIN / VALIDATION / TEST =====

train_df = pd.concat([datasets[file_name] for file_name in TRAIN_FILES], ignore_index=True)  # Ghép Data1, Data2 và Data3 thành tập Train.

validation_df = pd.concat([datasets[file_name] for file_name in VALIDATION_FILES], ignore_index=True)  # Tạo tập Validation từ Data4.

test_df = pd.concat([datasets[file_name] for file_name in TEST_FILES], ignore_index=True)  # Tạo tập Test từ Data5.

print('===== KÍCH THƯỚC DATASET =====')  # In tiêu đề kiểm tra kích thước.

print('Train:', train_df.shape)  # In kích thước tập Train.

print('Validation:', validation_df.shape)  # In kích thước tập Validation.

print('Test:', test_df.shape)  # In kích thước tập Test.

print('\nTổng số mẫu:', len(train_df) + len(validation_df) + len(test_df))  # In tổng số mẫu của cả ba tập.

===== KÍCH THƯỚC DATASET =====
Train: (90000, 17)
Validation: (30000, 17)
Test: (30000, 17)

Tổng số mẫu: 150000


In [ ]:
# ===== CELL 3: TÁCH INPUT VÀ TARGET =====

X_train = train_df[INPUT_COLUMNS].copy()  # Lấy 5 biến ud, uq, Id, Iq và nm làm input Train.

Y_train = train_df[TARGET_COLUMNS].copy()  # Lấy Rs, Ld và Lq làm target Train.

X_validation = validation_df[INPUT_COLUMNS].copy()  # Lấy 5 biến input cho Validation.

Y_validation = validation_df[TARGET_COLUMNS].copy()  # Lấy 3 target cho Validation.

X_test = test_df[INPUT_COLUMNS].copy()  # Lấy 5 biến input cho Test.

Y_test = test_df[TARGET_COLUMNS].copy()  # Lấy 3 target cho Test.

Time_train = train_df[[TIME_COLUMN]].copy()  # Giữ Time của Train để phục vụ physics loss về sau.

Time_validation = validation_df[[TIME_COLUMN]].copy()  # Giữ Time của Validation cho các bước PINN sau này.

Time_test = test_df[[TIME_COLUMN]].copy()  # Giữ Time của Test cho các bước PINN sau này.

print('X_train:', X_train.shape)  # In kích thước Input Train.

print('Y_train:', Y_train.shape)  # In kích thước Target Train.

print('X_validation:', X_validation.shape)  # In kích thước Input Validation.

print('Y_validation:', Y_validation.shape)  # In kích thước Target Validation.

print('X_test:', X_test.shape)  # In kích thước Input Test.

print('Y_test:', Y_test.shape)  # In kích thước Target Test.

print('\nInput:', INPUT_COLUMNS)  # In danh sách input.

print('Target:', TARGET_COLUMNS)  # In danh sách target.

X_train: (90000, 5)
Y_train: (90000, 3)
X_validation: (30000, 5)
Y_validation: (30000, 3)
X_test: (30000, 5)
Y_test: (30000, 3)

Input: ['ud', 'uq', 'Id', 'Iq', 'nm']
Target: ['Rs', 'Ld', 'Lq']


In [ ]:
# ===== CELL 4: KIỂM TRA DỮ LIỆU SAU KHI TÁCH FEATURE =====

for name, X, Y in [('Train', X_train, Y_train), ('Validation', X_validation, Y_validation), ('Test', X_test, Y_test)]:  # Lặp qua ba tập dữ liệu.

    print(f'\n===== {name} =====')  # In tên tập dữ liệu hiện tại.

    print('NaN trong Input:', X.isna().sum().sum())  # Đếm NaN trong Input.

    print('NaN trong Target:', Y.isna().sum().sum())  # Đếm NaN trong Target.

    print('Inf trong Input:', np.isinf(X.to_numpy()).sum())  # Đếm Inf trong Input.

    print('Inf trong Target:', np.isinf(Y.to_numpy()).sum())  # Đếm Inf trong Target.

    print('Input min:')  # In tiêu đề giá trị nhỏ nhất của Input.

    print(X.min())  # In giá trị nhỏ nhất của từng Input.

    print('Input max:')  # In tiêu đề giá trị lớn nhất của Input.

    print(X.max())  # In giá trị lớn nhất của từng Input.

    print('Target min:')  # In tiêu đề giá trị nhỏ nhất của Target.

    print(Y.min())  # In giá trị nhỏ nhất của từng Target.

    print('Target max:')  # In tiêu đề giá trị lớn nhất của Target.

    print(Y.max())  # In giá trị lớn nhất của từng Target.


===== Train =====
NaN trong Input: 0
NaN trong Target: 0
Inf trong Input: 0
Inf trong Target: 0
Input min:
ud   -1.444806e+02
uq    2.212863e-03
Id   -5.593699e+01
Iq   -2.278581e-01
nm    5.301870e-14
dtype: float64
Input max:
ud       1.102821
uq     173.205081
Id       2.750290
Iq      65.676215
nm    1001.363157
dtype: float64
Target min:
Rs    0.033828
Ld    0.001229
Lq    0.001305
dtype: float64
Target max:
Rs    0.044186
Ld    0.001360
Lq    0.001441
dtype: float64

===== Validation =====
NaN trong Input: 0
NaN trong Target: 0
Inf trong Input: 0
Inf trong Target: 0
Input min:
ud   -1.437154e+02
uq    2.212863e-03
Id   -5.594341e+01
Iq   -2.386857e-01
nm    5.301870e-14
dtype: float64
Input max:
ud       1.102821
uq     173.205081
Id       2.749125
Iq      65.641065
nm    1001.363722
dtype: float64
Target min:
Rs    0.033828
Ld    0.001229
Lq    0.001305
dtype: float64
Target max:
Rs    0.044186
Ld    0.001360
Lq    0.001441
dtype: float64

===== Test =====
NaN trong Input: 0
Na

In [ ]:
# ===== CELL 5: CHUẨN HÓA INPUT VÀ TARGET =====

from sklearn.preprocessing import StandardScaler  # Import StandardScaler để chuẩn hóa dữ liệu.

X_scaler = StandardScaler()  # Tạo scaler cho 5 biến Input.

Y_scaler = StandardScaler()  # Tạo scaler cho 3 biến Target.

X_train_scaled = X_scaler.fit_transform(X_train)  # Fit scaler bằng Input Train và chuẩn hóa Input Train.

X_validation_scaled = X_scaler.transform(X_validation)  # Dùng scaler của Train để chuẩn hóa Input Validation.

X_test_scaled = X_scaler.transform(X_test)  # Dùng scaler của Train để chuẩn hóa Input Test.

Y_train_scaled = Y_scaler.fit_transform(Y_train)  # Fit scaler bằng Target Train và chuẩn hóa Target Train.

Y_validation_scaled = Y_scaler.transform(Y_validation)  # Dùng scaler của Train để chuẩn hóa Target Validation.

Y_test_scaled = Y_scaler.transform(Y_test)  # Dùng scaler của Train để chuẩn hóa Target Test.

print('X_train_scaled:', X_train_scaled.shape)  # In kích thước Input Train sau chuẩn hóa.

print('X_validation_scaled:', X_validation_scaled.shape)  # In kích thước Input Validation sau chuẩn hóa.

print('X_test_scaled:', X_test_scaled.shape)  # In kích thước Input Test sau chuẩn hóa.

print('Y_train_scaled:', Y_train_scaled.shape)  # In kích thước Target Train sau chuẩn hóa.

print('Y_validation_scaled:', Y_validation_scaled.shape)  # In kích thước Target Validation sau chuẩn hóa.

print('Y_test_scaled:', Y_test_scaled.shape)  # In kích thước Target Test sau chuẩn hóa.

X_train_scaled: (90000, 5)
X_validation_scaled: (30000, 5)
X_test_scaled: (30000, 5)
Y_train_scaled: (90000, 3)
Y_validation_scaled: (30000, 3)
Y_test_scaled: (30000, 3)


In [ ]:
# ===== CELL 6: KIỂM TRA KẾT QUẢ CHUẨN HÓA =====

X_train_check = pd.DataFrame(X_train_scaled, columns=INPUT_COLUMNS)  # Chuyển Input Train đã chuẩn hóa thành DataFrame để dễ kiểm tra.

Y_train_check = pd.DataFrame(Y_train_scaled, columns=TARGET_COLUMNS)  # Chuyển Target Train đã chuẩn hóa thành DataFrame để dễ kiểm tra.

print('===== INPUT TRAIN SAU CHUẨN HÓA =====')  # In tiêu đề kiểm tra Input.

print(X_train_check.describe().T)  # In thống kê Input Train sau chuẩn hóa.

print('\n===== TARGET TRAIN SAU CHUẨN HÓA =====')  # In tiêu đề kiểm tra Target.

print(Y_train_check.describe().T)  # In thống kê Target Train sau chuẩn hóa.

===== INPUT TRAIN SAU CHUẨN HÓA =====
      count          mean       std       min       25%       50%       75%  \
ud  90000.0 -2.021099e-16  1.000006 -6.356558 -0.623699 -0.091050  0.980563   
uq  90000.0  1.111605e-16  1.000006 -2.219929 -1.107668  0.747369  0.864380   
Id  90000.0  1.212660e-16  1.000006 -1.926861  0.048799  0.557758  0.572093   
Iq  90000.0 -1.515825e-16  1.000006 -1.288802 -0.739906 -0.196572  0.875606   
nm  90000.0  1.818989e-16  1.000006 -1.635566 -1.006914  0.248804  0.700309   

         max  
ud  1.278287  
uq  1.500448  
Id  0.683750  
Iq  2.395401  
nm  1.512495  

===== TARGET TRAIN SAU CHUẨN HÓA =====
      count          mean       std       min       25%       50%       75%  \
Rs  90000.0  9.221266e-17  1.000006 -1.463483 -1.014853 -0.027801  0.929340   
Ld  90000.0  3.395447e-15  1.000006 -0.625914 -0.569690 -0.550873 -0.303326   
Lq  90000.0  2.379844e-15  1.000006 -3.320218 -0.078469  0.378234  0.460450   

         max  
Rs  1.634633  
Ld  1.8866

In [ ]:
# ===== CELL 7: LƯU DATASET ĐÃ PREPROCESSING VÀ SCALER =====

import joblib  # Import joblib để lưu scaler và sử dụng lại trong quá trình inference.

np.save(os.path.join(PROCESSED_DATA_DIR, 'X_train.npy'), X_train_scaled)  # Lưu Input Train đã chuẩn hóa.

np.save(os.path.join(PROCESSED_DATA_DIR, 'Y_train.npy'), Y_train_scaled)  # Lưu Target Train đã chuẩn hóa.

np.save(os.path.join(PROCESSED_DATA_DIR, 'X_validation.npy'), X_validation_scaled)  # Lưu Input Validation đã chuẩn hóa.

np.save(os.path.join(PROCESSED_DATA_DIR, 'Y_validation.npy'), Y_validation_scaled)  # Lưu Target Validation đã chuẩn hóa.

np.save(os.path.join(PROCESSED_DATA_DIR, 'X_test.npy'), X_test_scaled)  # Lưu Input Test đã chuẩn hóa.

np.save(os.path.join(PROCESSED_DATA_DIR, 'Y_test.npy'), Y_test_scaled)  # Lưu Target Test đã chuẩn hóa.

np.save(os.path.join(PROCESSED_DATA_DIR, 'Time_train.npy'), Time_train.to_numpy())  # Lưu thời gian của Train để phục vụ PINN.

np.save(os.path.join(PROCESSED_DATA_DIR, 'Time_validation.npy'), Time_validation.to_numpy())  # Lưu thời gian của Validation để phục vụ PINN.

np.save(os.path.join(PROCESSED_DATA_DIR, 'Time_test.npy'), Time_test.to_numpy())  # Lưu thời gian của Test để phục vụ PINN.

joblib.dump(X_scaler, os.path.join(PROCESSED_DATA_DIR, 'X_scaler.pkl'))  # Lưu scaler của Input để sử dụng lại khi validation, test và deployment.

joblib.dump(Y_scaler, os.path.join(PROCESSED_DATA_DIR, 'Y_scaler.pkl'))  # Lưu scaler của Target để chuyển output normalized về đơn vị vật lý.

print('Đã lưu preprocessing vào:', PROCESSED_DATA_DIR)  # In đường dẫn thư mục chứa toàn bộ dữ liệu đã xử lý.

Đã lưu preprocessing vào: /content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN/dataset/processed
